# 4. Memoria conversacional con LangGraph

## Objetivos de Aprendizaje
- Comprender la importancia de la memoria en conversaciones con LLMs
- Implementar memoria de hilo con la persistencia de LangGraph (`InMemorySaver`)
- Comparar tres estrategias: buffer, ventana y resumen
- Optimizar el uso de tokens con esas estrategias

## ¿Por qué es Importante la Memoria?

Los LLMs son **stateless** por naturaleza: no recuerdan conversaciones anteriores. La memoria permite:
- **Contexto conversacional**: Referirse a mensajes anteriores
- **Personalización**: Recordar preferencias del usuario
- **Continuidad**: Mantener hilos de conversación coherentes
- **Experiencia natural**: Conversaciones que se sienten humanas

En LangChain 1.x, `RunnableWithMessageHistory` está **deprecado**. El recambio oficial es la persistencia de LangGraph: un **checkpointer** guarda el estado y un **`thread_id`** identifica la sesión.

## Estrategias que vamos a comparar

1. **Buffer**: el modelo ve todo el historial
2. **Ventana**: el modelo ve solo los N intercambios más recientes
3. **Resumen**: los mensajes viejos se compactan en un resumen para ahorrar tokens


In [ ]:
# Importar bibliotecas necesarias para memoria
import os

# Carga de credenciales: funciona igual en Google Colab y en local (.env)
try:
    from google.colab import userdata  # type: ignore
    os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()

assert os.getenv("GROQ_API_KEY"), "Falta GROQ_API_KEY (Colab: Secrets · local: archivo .env)"

MODELO_RAPIDO = os.getenv("GROQ_MODEL_FAST", "openai/gpt-oss-20b")

from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    RemoveMessage,
    SystemMessage,
)
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, MessagesState, StateGraph
from langgraph.graph.message import REMOVE_ALL_MESSAGES

print("✓ Bibliotecas de memoria importadas correctamente")


In [ ]:
# Configuración del modelo para memoria
# Usamos el modelo rápido: los ejemplos de memoria hacen muchas llamadas cortas
# ChatGroq lee GROQ_API_KEY del entorno automáticamente
try:
    llm = ChatGroq(
        model=MODELO_RAPIDO,
        temperature=0.1,
        reasoning_effort="low",
    )

    print("✓ Modelo configurado para experimentos de memoria")
    print(f"Modelo: {llm.model_name}")

except Exception as e:
    print(f"✗ Error en configuración: {e}")
    print("Verifica la variable de entorno GROQ_API_KEY")


In [ ]:
# Un grafo de un nodo: el checkpointer ES la memoria.
# El mismo grafo atiende varias sesiones; las distingue `thread_id`.

def crear_chat_con_memoria(estrategia: str = "buffer", k: int = 2, max_mensajes: int = 6):
    """estrategia: 'buffer' | 'ventana' | 'resumen'."""

    def preparar(mensajes):
        if estrategia == "ventana":
            return mensajes[-(k * 2):], None
        if estrategia == "resumen" and len(mensajes) > max_mensajes:
            antiguos, recientes = mensajes[:-2], mensajes[-2:]
            texto = "\n".join(
                f"{'Usuario' if m.type == 'human' else 'Asistente'}: {m.content}"
                for m in antiguos
            )
            resumen = llm.invoke(
                f"Resume esta conversación en 2-3 líneas:\n{texto}"
            ).content
            visibles = [AIMessage(content=f"[RESUMEN]: {resumen}"), *recientes]
            return visibles, visibles
        return mensajes, None

    def nodo(state: MessagesState):
        visibles, reemplazo = preparar(state["messages"])
        respuesta = llm.invoke(
            [SystemMessage(content="Eres un asistente útil."), *visibles]
        )
        if reemplazo is not None:
            return {
                "messages": [
                    RemoveMessage(id=REMOVE_ALL_MESSAGES),
                    *reemplazo,
                    respuesta,
                ]
            }
        return {"messages": [respuesta]}

    grafo = StateGraph(MessagesState)
    grafo.add_node("modelo", nodo)
    grafo.add_edge(START, "modelo")
    return grafo.compile(checkpointer=InMemorySaver())


def cfg(thread_id: str) -> dict:
    return {"configurable": {"thread_id": thread_id}}


def preguntar(app, thread_id: str, texto: str):
    salida = app.invoke({"messages": [HumanMessage(content=texto)]}, cfg(thread_id))
    return salida["messages"][-1]


def historial(app, thread_id: str) -> list:
    snap = app.get_state(cfg(thread_id))
    return list(snap.values.get("messages", [])) if snap.values else []


print("✓ Helper de memoria con LangGraph listo (sin RunnableWithMessageHistory)")


## 1. Buffer — memoria completa

El checkpointer guarda **todo** el historial del `thread_id`. Es la estrategia más simple, pero el consumo de tokens crece con cada turno.


In [ ]:
def ejemplo_buffer_memory():
    print("=== BUFFER (historial completo) ===")
    print("El modelo ve todos los mensajes del hilo\n")

    app = crear_chat_con_memoria("buffer")
    thread_id = "demo_buffer"

    try:
        print("1. Primera pregunta:")
        r1 = preguntar(app, thread_id, "Mi nombre es Ana y soy programadora Python")
        print(f"Respuesta: {r1.content}\n")

        print("2. Segunda pregunta:")
        r2 = preguntar(app, thread_id, "¿Cuál es mi nombre y profesión?")
        print(f"Respuesta: {r2.content}\n")

        print("3. Tercera pregunta:")
        r3 = preguntar(app, thread_id, "¿Qué lenguaje de programación mencioné?")
        print(f"Respuesta: {r3.content}\n")

        print("=== CONTENIDO DE LA MEMORIA ===")
        for i, msg in enumerate(historial(app, thread_id), 1):
            print(f"{i}. {msg.type}: {msg.content}")

    except Exception as e:
        print(f"Error: {e}")

ejemplo_buffer_memory()


## 2. Ventana — solo los N intercambios recientes

El checkpointer sigue guardando todo (útil para depurar). Al llamar al modelo, recortamos a los últimos `k` intercambios (`k*2` mensajes).


In [ ]:
K = 2  # intercambios visibles (usuario + asistente)

def ejemplo_window_memory():
    print(f"=== VENTANA (k={K}) ===")
    print("El modelo solo ve los 2 intercambios más recientes\n")

    app = crear_chat_con_memoria("ventana", k=K)
    thread_id = "demo_ventana"
    entradas = [
        "Mi nombre es Carlos y tengo 30 años",
        "Trabajo como diseñador gráfico",
        "Me gusta el café y la música jazz",
        "¿Puedes recordar mi edad?",
        "¿Cuál es mi profesión?",
    ]

    try:
        for i, user_input in enumerate(entradas, 1):
            print(f"{'='*20} INTERACCIÓN {i} {'='*20}")
            print(f"👤 Usuario: {user_input}")

            respuesta = preguntar(app, thread_id, user_input)
            print(f"🤖 Asistente: {respuesta.content}\n")

            todos = historial(app, thread_id)
            visibles = todos[-(K * 2):]
            print("📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total almacenado: {len(todos)} mensajes")
            print(f"   👁️  Visible al modelo: {len(visibles)} mensajes")
            print(f"   🗑️  Fuera de la ventana: {len(todos) - len(visibles)}")

            print(f"\n📚 HISTORIAL COMPLETO ({len(todos)} mensajes):")
            for j, msg in enumerate(todos, 1):
                rol = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                contenido = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                en_ventana = j > len(todos) - len(visibles)
                marca = "✅" if en_ventana else "❌"
                print(f"     {j}. {marca} {rol}: {contenido}")

            print(f"\n🔍 VENTANA VISIBLE AL MODELO ({len(visibles)} mensajes):")
            for j, msg in enumerate(visibles, 1):
                rol = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                contenido = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
                print(f"     {j}. ✅ {rol}: {contenido}")

            print("\n" + "=" * 60 + "\n")

    except Exception as e:
        print(f"Error: {e}")

ejemplo_window_memory()


## 3. Resumen — compactar el pasado

Cuando el hilo supera un umbral, los mensajes viejos se sustituyen por un resumen generado por el propio LLM. Así el modelo conserva el contexto largo sin reenviar todo el texto.


In [ ]:
def ejemplo_summary_memory():
    print("=== RESUMEN ===")
    print("Resume conversaciones largas para ahorrar tokens\n")

    app = crear_chat_con_memoria("resumen", max_mensajes=6)
    thread_id = "demo_resumen"
    entradas = [
        "Hola, soy María González, ingeniera de software de 35 años",
        "Trabajo en una startup de fintech en Madrid desarrollando pagos digitales",
        "Usamos React, Node.js, Docker y Kubernetes en nuestros proyectos",
        "Mi mayor desafío es la latencia en transacciones internacionales",
        "También trabajo en mejorar la UX de nuestra app móvil",
        "¿Puedes resumir quién soy y cuáles son mis principales desafíos?",
    ]

    try:
        for i, user_input in enumerate(entradas, 1):
            print(f"{'='*15} INTERACCIÓN {i} {'='*15}")
            print(f"👤 Usuario: {user_input}")

            respuesta = preguntar(app, thread_id, user_input)
            print(f"🤖 Asistente: {respuesta.content}\n")

            msgs = historial(app, thread_id)
            tiene_resumen = any("[RESUMEN]" in (msg.content or "") for msg in msgs)
            print("📊 ESTADO DE LA MEMORIA:")
            print(f"   💾 Total mensajes: {len(msgs)}")
            print(f"   📝 Tiene resumen: {'✅ Sí' if tiene_resumen else '❌ No'}")

            print("\n💬 CONTENIDO ACTUAL DE LA MEMORIA:")
            for j, msg in enumerate(msgs, 1):
                contenido = msg.content or ""
                if "[RESUMEN]" in contenido:
                    rol = "📝 Resumen"
                    contenido = contenido.replace("[RESUMEN]: ", "")
                else:
                    rol = "👤 Usuario" if msg.type == "human" else "🤖 Asistente"
                if len(contenido) > 80:
                    contenido = contenido[:80] + "..."
                print(f"   {j}. {rol}: {contenido}")

            print("\n" + "=" * 50 + "\n")

    except Exception as e:
        print(f"Error: {e}")

ejemplo_summary_memory()


## Consideraciones Técnicas y Mejores Prácticas

### Selección de estrategia

| Tipo | Cuándo usarlo | Ventajas | Desventajas |
|------|---------------|----------|-------------|
| **Buffer** | Conversaciones cortas | Contexto completo | Alto consumo de tokens |
| **Ventana** | Solo importa lo reciente | Eficiente en tokens | Puede olvidar datos clave |
| **Resumen** | Conversaciones largas | Balance eficiencia/contexto | Pierde detalles específicos |

El `InMemorySaver` vive en RAM: al reiniciar el kernel se pierde. En producción se cambia por un checkpointer persistente (SQLite, Postgres) sin tocar el grafo.

### Mejores prácticas

1. **Gestión de tokens**: no reenvíes historiales que no aportan al siguiente turno.
2. **Un `thread_id` por conversación**: así varios usuarios no mezclan memoria.
3. **Producción**: sustituye `InMemorySaver` por un checkpointer durable.

## Conceptos clave

1. Los LLMs no recuerdan: hay que **persistir y reinyectar** el historial
2. LangGraph guarda ese estado con un **checkpointer** y lo aísla por **`thread_id`**
3. Buffer, ventana y resumen son **estrategias** sobre el mismo mecanismo
4. Hay que equilibrar contexto y costo de tokens

## Conclusión del módulo IL1.1

Has completado la introducción a LLMs y conexiones API:

1. **APIs directas** vs **frameworks** como LangChain
2. **Streaming** para mejor experiencia de usuario
3. **Memoria** de hilo con LangGraph
4. **Mejores prácticas** de seguridad y de uso de tokens

### Próximos pasos
En **IL1.2** exploraremos técnicas de **prompt engineering**: zero-shot, few-shot y chain-of-thought.
